## 0. Assignment Overview

The assignment examines trip generation using three complementary approaches:

1. Develop and compare household-based and zone-based linear regression models for daily work trips.
2. Develop a household-based linear regression model for daily non-work trips and compare it with the household work-trip model.
3. Develop a cross-classification model for daily non-work trips using household size and vehicle ownership.

This first stage is limited to project setup, data loading, and basic validation. Modeling and exploratory analysis will be developed in later stages.

## 1. Setup and Imports

In [ ]:
# report: hide-cell 
from pathlib import Path

import numpy as np
import pandas as pd

### Project Paths

The project root is identified from the current working directory so that the notebook can run from either the repository root or the `notebooks` directory.

In [ ]:
# report: hide-cell
working_directory = Path.cwd().resolve()
project_candidates = (working_directory, working_directory.parent)

project_root = next(
    (path for path in project_candidates if (path / "data" / "raw" / "ps1-data.csv").is_file()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate data/raw/ps1-data.csv from the current working directory."
    )

data_path = project_root / "data" / "raw" / "ps1-data.csv"

print(f"Project root: {project_root}")
print(f"Data file: {data_path.relative_to(project_root)}")

Project root: C:\Users\Student\OneDrive - University of Illinois Chicago\01_academic\508\Transportation_planning\508_transportation_planning
Data file: data\raw\ps1-data.csv


## 2. Data Loading

Each row represents one surveyed household. The expansion factor (`expfac`) is retained as provided but is not applied during this initial inspection.

In [ ]:
# report: hide-cell
households = pd.read_csv(data_path)
households.head()

,zone,dwtype,npers,nveh,nlic,nftw,nptw,nwah,nstud,nfem,nmale,nchild,n65+,nwork,nnwk,expfac
0,29,2,3,1,3,1,1,0,1,2,1,0,0,2,9,22.11
1,6,2,2,1,2,2,0,0,0,1,1,0,0,3,7,30.35
2,11,2,1,0,0,0,0,0,0,1,0,0,1,0,0,25.53
3,14,2,1,0,1,0,1,0,1,1,0,0,0,0,3,22.11
4,24,2,2,0,0,0,0,0,2,1,1,0,0,0,4,30.35


## 3. Data Inspection and Validation

The checks below confirm the dataset dimensions, variable names, data types, memory use, and missing-value counts.

In [ ]:
# report: hide-cell
print(f"Rows: {households.shape[0]:,}")
print(f"Columns: {households.shape[1]}")

Rows: 2,310
Columns: 16


In [5]:
pd.DataFrame({"column": households.columns})

,column
0,zone
1,dwtype
2,npers
3,nveh
4,nlic
5,nftw
6,nptw
7,nwah
8,nstud
9,nfem


In [6]:
households.dtypes.rename("dtype").to_frame()

,dtype
zone,int64
dwtype,int64
npers,int64
nveh,int64
nlic,int64
nftw,int64
nptw,int64
nwah,int64
nstud,int64
nfem,int64


In [7]:
households.info()

<class 'pandas.DataFrame'>
RangeIndex: 2310 entries, 0 to 2309
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   zone    2310 non-null   int64  
 1   dwtype  2310 non-null   int64  
 2   npers   2310 non-null   int64  
 3   nveh    2310 non-null   int64  
 4   nlic    2310 non-null   int64  
 5   nftw    2310 non-null   int64  
 6   nptw    2310 non-null   int64  
 7   nwah    2310 non-null   int64  
 8   nstud   2310 non-null   int64  
 9   nfem    2310 non-null   int64  
 10  nmale   2310 non-null   int64  
 11  nchild  2310 non-null   int64  
 12  n65+    2310 non-null   int64  
 13  nwork   2310 non-null   int64  
 14  nnwk    2310 non-null   int64  
 15  expfac  2310 non-null   float64
dtypes: float64(1), int64(15)
memory usage: 288.9 KB


In [8]:
missing_values = (
    households.isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)
missing_values

,missing_values
zone,0
dwtype,0
npers,0
nveh,0
nlic,0
nftw,0
nptw,0
nwah,0
nstud,0
nfem,0


In [9]:
expected_columns = [
    "zone", "dwtype", "npers", "nveh", "nlic", "nftw", "nptw",
    "nwah", "nstud", "nfem", "nmale", "nchild", "n65+", "nwork",
    "nnwk", "expfac",
]

assert not households.empty, "The dataset is empty."
assert households.columns.tolist() == expected_columns, "Unexpected dataset columns."

print("Validation passed: the dataset is non-empty and contains all 16 expected columns.")

Validation passed: the dataset is non-empty and contains all 16 expected columns.
